# 01. Phân tích và xử lý dữ liệu

Đọc dữ liệu gốc (`data/raw/`), làm sạch, hợp nhất thành **một bảng ngày**, kiểm tra chất lượng và phân tích quan hệ giữa các biến.
Notebook này **không** điền thiếu, không chuẩn hoá, không chia tập: bảng `data/processed/cuadat_huana_mua_daily.csv` giữ nguyên
ngày thiếu (NaN) cùng cờ thiếu; mọi bước tiền xử lý cho mô hình nằm trong `src/data/` và được dựng một lần ở notebook 02.

Mã nguồn: `src/data/quality_control.py`.

## 0. Chuẩn bị trên Google Colab (chạy trên máy thì bỏ qua mục này)

Code lấy từ GitHub, còn dữ liệu và kết quả lấy từ Google Drive. Cả hai được đặt vào **cùng một thư mục trên Drive**
(`MyDrive/T_DAThS_Ninh`), nên kết quả không mất khi tắt Colab và các notebook 02–10 dùng chung thư mục này.

1. Tải hai file nén lên **My Drive** (thư mục gốc):
   - `Ninh_data.zip` (bắt buộc): chứa thư mục `data/`;
   - `Ninh_results.zip` (không bắt buộc): chứa thư mục `results/` (kết quả và weight đã huấn luyện). Không có file này thì
     kết quả được sinh ra khi chạy notebook 03–09.
2. **Runtime → Change runtime type → GPU (T4)**.
3. Chạy ô 0a (lấy code), ô 0b (giải nén), rồi chạy tiếp các ô bên dưới như bình thường.

Chỉ cần làm mục này **một lần**. Các notebook khác chỉ cần mở và chạy: ô đầu tiên của chúng tự gắn Drive và tìm thư mục dự án.

In [ ]:
# ── 0a. (Colab) Lấy code từ GitHub về Google Drive. Chạy trên máy thì tự bỏ qua. ──
import os, sys, subprocess
REPO  = 'https://github.com/ductrunglyn/T_DAThS_Ninh.git'
DU_AN = '/content/drive/MyDrive/T_DAThS_Ninh'          # thư mục dự án trên Drive: code + data/ + results/
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.isdir(os.path.join(DU_AN, '.git')):      # đã clone trước đó → cập nhật code mới nhất
        r = subprocess.run(['git', '-C', DU_AN, 'pull', '--ff-only'], capture_output=True, text=True)
        print(r.stdout or r.stderr)
        if r.returncode:
            print('Không cập nhật được (có file đã sửa trên Drive): giữ nguyên code hiện tại.')
    elif os.path.isdir(os.path.join(DU_AN, 'src')):     # thư mục dự án chép tay lên Drive, không phải bản clone
        print('Đã có code ở', DU_AN, '(không phải bản clone): dùng luôn.')
    else:
        subprocess.run(['git', 'clone', REPO, DU_AN], check=True)
    os.chdir(DU_AN)
    print('Thư mục dự án:', DU_AN); print(sorted(os.listdir(DU_AN)))
else:
    print('Không chạy trên Colab: bỏ qua.')

In [ ]:
# ── 0b. (Colab) Giải nén data và kết quả (file .zip đã tải lên Drive) vào thư mục dự án. Chạy trên máy thì tự bỏ qua. ──
import zipfile
FILE_NEN = {'data':    '/content/drive/MyDrive/Ninh_data.zip',       # bắt buộc
            'results': '/content/drive/MyDrive/Ninh_results.zip'}    # không bắt buộc
GHI_DE = False      # True: giải nén lại dù thư mục đã có (ghi đè các file cùng tên)
if 'google.colab' in sys.modules:
    for ten, f in FILE_NEN.items():
        dich = os.path.join(DU_AN, ten)
        if os.path.isdir(dich) and os.listdir(dich) and not GHI_DE:
            print(f'{ten}/ đã có: bỏ qua (đặt GHI_DE = True để giải nén lại).')
        elif os.path.isfile(f):
            with zipfile.ZipFile(f) as z:
                ds = [n for n in z.namelist() if not n.startswith('__MACOSX')]
                ca_thu_muc = all(n.startswith(ten + '/') for n in ds)          # nén cả thư mục hay chỉ phần bên trong
                z.extractall(DU_AN if ca_thu_muc else dich, members=ds)
            print(f'Đã giải nén {f} → {dich}')
        elif ten == 'data':
            raise FileNotFoundError(f'Không thấy {f}: tải Ninh_data.zip lên My Drive, hoặc sửa đường dẫn trong FILE_NEN.')
        else:
            print(f'Không thấy {f}: bỏ qua. Kết quả sẽ được sinh ra khi chạy notebook 03–09.')
else:
    print('Không chạy trên Colab: bỏ qua.')

## Thiết lập

In [ ]:
# ── Google Colab: gắn Google Drive, vào thư mục dự án, cài đúng phiên bản thư viện. Chạy trên máy thì tự bỏ qua. ──
import os, sys, glob, subprocess
DU_AN = '/content/drive/MyDrive/T_DAThS_Ninh'   # ← thư mục dự án trên Google Drive (notebook 01, mục 0 tạo ra)
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir(os.path.join(DU_AN, 'src')):          # không thấy → tự tìm trong My Drive (sâu tối đa 3 cấp)
        hit = [h for k in range(1, 4) for h in glob.glob('/content/drive/MyDrive/' + '*/' * k + 'src/results_store.py')]
        assert hit, ('Chưa có dự án trên Google Drive: mở notebook 01_PhanTich_XuLy_DuLieu và chạy 2 ô của mục 0 '
                     '(lấy code từ GitHub, giải nén data/kết quả), hoặc sửa biến DU_AN ở đầu ô này')
        DU_AN = os.path.dirname(os.path.dirname(hit[0]))
    os.chdir(DU_AN)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements_colab.txt'], check=True)

import json, warnings
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src' / 'results_store.py').exists():
    assert ROOT != ROOT.parent, 'Không tìm thấy thư mục dự án (thư mục chứa src/): mở notebook từ trong dự án'
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 80)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})
from src.utils import load_config, results_dir, dataset_meta
cfg = load_config()
print('Thư mục dự án:', ROOT, '| kết quả lưu ở:', cfg['paths']['results'])
print('Thiết bị:', f'GPU {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else
      'CPU (trên Colab: Runtime → Change runtime type → GPU để chạy nhanh hơn)')
from src.data import quality_control as qc
OUT = results_dir(cfg, 'phan_tich_du_lieu')

## 1. Đọc, làm sạch và hợp nhất

Giai đoạn chung của hai hồ 01/01/2019 – 12/08/2026 (bỏ 13/08/2026 vì số giờ phát điện bằng 0 ở cả hai hồ). Mưa CHIRPS trung bình
vùng: Vùng 1 — lưu vực Hủa Na; Vùng 2 — khu giữa Hủa Na và Cửa Đạt.

In [ ]:
df, log = qc.build_daily(cfg)
for l in log: print('•', l)
f = ROOT / cfg['paths']['processed'] / 'cuadat_huana_mua_daily.csv'
f.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(f, encoding='utf-8-sig')
print(f'\nĐã lưu {f.relative_to(ROOT)}: {df.shape[0]} ngày × {df.shape[1]} cột; Cửa Đạt thiếu {df.cd_missing.sum()}, Hủa Na thiếu {df.hn_missing.sum()} ngày')
display(df.describe().T.round(2))

## 2. Ngày thiếu theo độ dài khoảng thiếu

Phân loại phục vụ thiết kế điền thiếu nhân quả (notebook 02): 1 ngày, 2–3 ngày, 4–7 ngày, > 7 ngày.

In [ ]:
gaps = qc.gap_table(df)
tab = gaps.groupby(['trạm', 'loại'])['số ngày'].agg(['count', 'sum']).rename(columns={'count': 'số đoạn', 'sum': 'số ngày'})
display(tab); gaps.to_csv(OUT / 'khoang_thieu.csv', index=False, encoding='utf-8-sig')
print('Khoảng thiếu > 3 ngày:'); display(gaps[gaps['số ngày'] > 3])
print('Số ngày thiếu theo năm:')
display(pd.DataFrame({'Cửa Đạt': df.cd_missing.groupby(df.index.year).sum(), 'Hủa Na': df.hn_missing.groupby(df.index.year).sum()}).T)

## 3. Kiểm tra chất lượng dữ liệu

### 3.1 Tổng hợp
1. Khớp nguồn. 2. Căn thời gian mưa: tương quan mưa(t − k) với *mức tăng* log Q CĐ(t). 3. Lan truyền Hủa Na → Cửa Đạt ở thang
ngày. 4. Nhiễu "răng cưa" của Q đến CĐ. 5. Đoạn giá trị không đổi ≥ 5 ngày.

In [ ]:
audit = qc.quality_audit(cfg, df)
display(audit); audit.to_csv(OUT / 'kiem_tra_chat_luong.csv', encoding='utf-8-sig')

### 3.1b Giờ phát điện và các ô bị đặt thành NaN

Số giờ phát điện mâu thuẫn với Q qua nhà máy ở nhiều ngày → biến `hn_gio` có độ tin cậy thấp (chỉ dùng ở tập phụ F4).
Bảng thứ hai liệt kê **mọi** ô bị đặt NaN hoặc thiếu so với lịch ngày liên tục.

In [ ]:
hrs = qc.operation_hours_check(cfg); display(hrs); hrs.to_csv(OUT / 'gio_van_hanh.csv', encoding='utf-8-sig')
nanr = qc.nan_replacements(cfg, df); nanr.to_csv(OUT / 'o_dat_nan.csv', index=False, encoding='utf-8-sig')
display(nanr.groupby(['nguồn', 'cột', 'lý do']).size().rename('số ô').to_frame())

### 3.2 Giá trị lặp của Q đến Hủa Na

Không tự động coi đoạn lặp là lỗi. Trong bộ dữ liệu chuẩn có hai biến **nhân quả** (chỉ dùng dữ liệu ≤ t):
`qhn_rep_len` = số ngày liên tiếp tính đến t mà Q đến HN bằng giá trị ngày trước; `qhn_rep_flag` = 1 khi chuỗi lặp đã kéo dài
≥ 5 ngày. Không có thông tin phân loại chất lượng từ nguồn số liệu nên không tạo biến `qhn_data_quality`.

In [ ]:
rep = qc.repeat_runs(df.hn_qden, cfg['quality']['repeat_min'])
print(f'{len(rep)} đoạn, {rep["số ngày"].sum()} ngày; theo tháng bắt đầu:', rep['tháng'].value_counts().sort_index().to_dict())
print('Giá trị lặp phổ biến:', rep['giá trị'].round(2).value_counts().head(8).to_dict())
display(rep.sort_values('số ngày', ascending=False).head(10)); rep.to_csv(OUT / 'doan_lap_q_den_hn.csv', index=False, encoding='utf-8-sig')

### 3.3 Nhiễu "răng cưa" của Q đến Cửa Đạt — đối chiếu nguyên nhân

Ngày răng cưa: Q tăng (giảm) > 30 % rồi giảm (tăng) > 30 % ngay ngày sau. Mỗi ngày được đối chiếu với mưa hai vùng (t−2 … t),
thay đổi Q ra Hủa Na, thay đổi mực nước và Q xả Cửa Đạt. Ngày **không có mưa ≥ 10 mm và Q ra Hủa Na đổi < 35 %** được đánh dấu
"nghi nhiễu tính toán". Tên nhãn được giữ từ trước khi có xác nhận: đơn vị cung cấp đã xác nhận Q đến hồ là số liệu đo
(13/11/2026, VX-02), nên trong báo cáo các ngày này được gọi là "răng cưa chưa rõ nguyên nhân".
**Không sửa và không làm trơn nhãn**: chỉ báo cáo. Đầu vào có thêm biến trung bình trượt mũ nhân quả `qcd_ema3`, được kiểm tra
như một thực nghiệm phụ (P_EMA).

In [ ]:
saw = qc.sawtooth_table(df, cfg['quality']['sawtooth_jump'])
saw.to_csv(OUT / 'rang_cua_q_cd.csv', index=False, encoding='utf-8-sig')
print(saw['đánh giá'].value_counts().to_string())
print('\nTheo hướng:'); display(pd.crosstab(saw['hướng'], saw['đánh giá']))
print('Theo tháng:'); display(pd.crosstab(pd.to_datetime(saw['ngày']).dt.month, saw['đánh giá']).T)
g = saw.groupby('đánh giá')[['Q(t)', 'ΔZ CĐ (t) cm', 'ΔZ CĐ (t+1) cm', 'ΔQ xả CĐ (t)']].agg(lambda s: s.abs().median())
print('Trung vị |giá trị| theo nhóm:'); display(g.round(1))
display(saw.head(8))

### 3.4 Đồng bộ thời gian giữa mưa CHIRPS và số liệu vận hành

Từ các file dữ liệu **không xác định được** quy ước ngày của từng nguồn: CHIRPS là tổng mưa theo ngày lịch của sản phẩm, còn số
liệu vận hành hồ là trung bình ngày theo quy ước báo cáo của đơn vị vận hành (có thể 0 h – 24 h hoặc 7 h – 7 h giờ địa phương).
Đơn vị cung cấp đã xác nhận cả hai nguồn đều theo ngày lịch (13/11/2026, VX-01, VX-06). Kiểm tra thực nghiệm dưới đây chỉ cho biết **không có độ lệch nguyên ngày** rõ ràng; nó
không loại trừ được độ lệch dưới 1 ngày (ví dụ 7 giờ).

In [ ]:
dq = np.log1p(df.q_cd).diff(); ok = ~df.cd_missing & ~df.cd_missing.shift(1, fill_value=True)
rows = []
for v in ['mua_v1', 'mua_v2']:
    cc = qc.lag_corr(df[v], dq, ok, range(-3, 4))
    rows.append({'biến': v, **{f'k = {k}': c for k, c in cc.items()}})
lagr = pd.DataFrame(rows).set_index('biến'); display(lagr.round(3))
# theo trận mưa lớn: ngày Q bắt đầu tăng so với ngày mưa lớn nhất
ev = []
for v in ['mua_v1', 'mua_v2']:
    big = df[v][(df[v] >= 40)].index
    for d in big:
        w = df.q_cd.loc[d - pd.Timedelta(days=1):d + pd.Timedelta(days=3)]
        if w.isna().any() or len(w) < 5: continue
        ev.append({'vùng': v, 'ngày mưa': d.date(), 'mưa': df.loc[d, v], 'ngày Q tăng mạnh nhất (so với ngày mưa)': int((w.diff().idxmax() - d).days)})
ev = pd.DataFrame(ev)
print('Các ngày mưa ≥ 40 mm: ngày có mức tăng Q lớn nhất trong (−1 … +3):')
display(ev.groupby('vùng')['ngày Q tăng mạnh nhất (so với ngày mưa)'].value_counts().unstack(fill_value=0))
lagr.to_csv(OUT / 'dong_bo_thoi_gian_mua.csv', encoding='utf-8-sig')

## 4. Phân tích quan hệ giữa các biến

Để không "nhìn" trước giai đoạn kiểm tra, tương quan và độ trễ chỉ tính trên **2019–2023** (ngày có quan trắc thật).

### 4.1 Chuỗi thời gian theo giai đoạn

In [ ]:
P1_END, P2_END, P3_START = pd.Timestamp('2023-12-31'), pd.Timestamp('2024-12-31'), pd.Timestamp('2025-01-01')
PER = pd.Series(np.where(df.index <= P1_END, '2019–2023', np.where(df.index <= P2_END, '2024', '2025–2026')), index=df.index)
fig, axs = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
axs[0].plot(df.q_cd, lw=.7); axs[0].set_ylabel('Q đến Cửa Đạt\n(m³/s)')
axs[1].plot(df.hn_qden, lw=.7, color='C7', label='Q đến Hủa Na'); axs[1].plot(df.hn_qxa, lw=.7, color='C1', label='Q ra Hủa Na')
axs[1].set_ylabel('Hủa Na\n(m³/s)'); axs[1].legend(fontsize=8, loc='upper left')
axs[2].plot(df.hn_mn, lw=.7, color='C5'); axs[2].set_ylabel('Mực nước\nHủa Na (m)')
axs[3].bar(df.index, df.mua_v1, width=1, color='C2', alpha=.7, label='Vùng 1 (lưu vực Hủa Na)')
axs[3].bar(df.index, -df.mua_v2, width=1, color='C9', alpha=.7, label='Vùng 2 (giữa 2 hồ), vẽ âm')
axs[3].set_ylabel('Mưa\n(mm/ngày)'); axs[3].legend(fontsize=8, loc='lower left')
for ax in axs:
    ax.axvspan(pd.Timestamp('2024-01-01'), P2_END, color='C4', alpha=.08); ax.axvspan(P3_START, df.index[-1], color='C3', alpha=.08)
plt.tight_layout(); plt.savefig(OUT / 'fig01_chuoi_thoi_gian.png'); plt.show()
real = ~df.cd_missing
st = df[real].groupby(PER[real])['q_cd'].describe(percentiles=[.5, .95, .99]).loc[['2019–2023', '2024', '2025–2026']]
display(st.round(1)); st.to_csv(OUT / 'phan_bo_q_theo_giai_doan.csv', encoding='utf-8-sig')

### 4.2 Tương quan (2019–2023, thang log)

In [ ]:
an = df.copy()
for v in ['v1', 'v2']: an[f'mua_{v}_7'] = an[f'mua_{v}'].rolling(7, min_periods=7).sum()
COR = {'q_cd': 'Q đến CĐ', 'hn_qden': 'Q đến HN', 'hn_qxa': 'Q ra HN', 'hn_qnm': 'Q nhà máy HN', 'hn_qtran': 'Q tràn HN',
       'hn_mn': 'Mực nước HN', 'mua_v1': 'Mưa V1', 'mua_v2': 'Mưa V2', 'mua_v1_7': 'Mưa V1 7 ngày', 'mua_v2_7': 'Mưa V2 7 ngày'}
trd = an.loc[(an.index <= P1_END) & ~df.cd_missing & ~df.hn_missing, list(COR)].dropna()
lg = [c for c in COR if c != 'hn_mn']; trd[lg] = np.log1p(trd[lg])
cor = trd.corr().rename(index=COR, columns=COR); display(cor.round(2)); cor.to_csv(OUT / 'tuong_quan_bien.csv', encoding='utf-8-sig')
fig, ax = plt.subplots(figsize=(8, 6.5)); im = ax.imshow(cor.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(cor))); ax.set_xticklabels(cor.columns, rotation=60, ha='right', fontsize=8)
ax.set_yticks(range(len(cor))); ax.set_yticklabels(cor.index, fontsize=8)
for i in range(len(cor)):
    for j in range(len(cor)): ax.text(j, i, f'{cor.values[i, j]:.2f}', ha='center', va='center', fontsize=7, color='w' if abs(cor.values[i, j]) > .6 else 'k')
plt.colorbar(im, ax=ax, shrink=.8); ax.grid(False); plt.tight_layout(); plt.savefig(OUT / 'fig02_tuong_quan.png'); plt.show()

### 4.3 Mực nước hồ Hủa Na và dòng chảy ra

In [ ]:
rd = df.loc[~df.hn_missing].copy()
rd['nhóm mực nước (m)'] = pd.cut(rd.hn_mn, [0, 225, 230, 235, 238, 241], labels=['< 225', '225–230', '230–235', '235–238', '≥ 238'], right=False)
lv = rd.groupby('nhóm mực nước (m)', observed=True).agg(**{'số ngày': ('hn_mn', 'size'), 'Q đến TB': ('hn_qden', 'mean'),
     'Q ra TB': ('hn_qxa', 'mean'), 'Q tràn TB': ('hn_qtran', 'mean'), '% ngày xả tràn > 50': ('hn_qtran', lambda s: 100 * (s > 50).mean())})
lv['Q ra / Q đến'] = lv['Q ra TB'] / lv['Q đến TB']; display(lv.round(2)); lv.to_csv(OUT / 'muc_nuoc_huana.csv', encoding='utf-8-sig')
fig, ax = plt.subplots(figsize=(7, 4)); sc = ax.scatter(rd.hn_mn, rd.hn_qxa, c=np.log1p(rd.hn_qden), s=5, cmap='viridis')
ax.set_yscale('log'); ax.set_xlabel('Mực nước Hủa Na (m)'); ax.set_ylabel('Q ra Hủa Na (m³/s)'); plt.colorbar(sc, ax=ax, label='log(1 + Q đến HN)')
plt.tight_layout(); plt.savefig(OUT / 'fig03_muc_nuoc_huana.png'); plt.show()

### 4.4 Độ trễ (2019–2023)

Tương quan giữa X(t − k) và Q đến CĐ(t), k = 0 … 10; tương quan riêng phần của Q ra Hủa Na sau khi loại ảnh hưởng của mưa hai vùng.
Đề cương mô tả độ trễ của Q và mưa trong phạm vi vài ngày; độ trễ cửa sổ L ∈ {3, 7, 14, 30} được chọn bằng CV ở các notebook mô hình.

In [ ]:
tr = an.loc[:P1_END]; y = np.log1p(tr.q_cd)
Zc = pd.concat([np.log1p(tr[f'mua_{v}']).shift(j) for v in ['v1', 'v2'] for j in range(4)] + [np.log1p(tr.mua_v1_7), np.log1p(tr.mua_v2_7)], axis=1)
def resid(v, Z):
    Z1 = np.c_[np.ones(len(Z)), Z]; return v - Z1 @ np.linalg.lstsq(Z1, v, rcond=None)[0]
SER = {'Q ra HN': np.log1p(tr.hn_qxa), 'Q đến HN': np.log1p(tr.hn_qden), 'Mực nước HN': tr.hn_mn, 'Mưa V1': np.log1p(tr.mua_v1), 'Mưa V2': np.log1p(tr.mua_v2)}
rows = []
for k in range(11):
    ok = (~tr.cd_missing) & ~tr.hn_missing.shift(k, fill_value=True).astype(bool) & Zc.notna().all(axis=1)
    row = {'trễ k': k, 'n': int(ok.sum()), **{f'r {nm}': np.corrcoef(s.shift(k)[ok], y[ok])[0, 1] for nm, s in SER.items()}}
    xx, yy, ZZ = SER['Q ra HN'].shift(k)[ok].values, y[ok].values, Zc[ok].values
    row['r riêng phần Q ra HN'] = np.corrcoef(resid(xx, ZZ), resid(yy, ZZ))[0, 1]; rows.append(row)
lag = pd.DataFrame(rows).set_index('trễ k'); display(lag.round(3)); lag.to_csv(OUT / 'do_tre_tuong_quan.csv', encoding='utf-8-sig')
fig, ax = plt.subplots(figsize=(9, 3.8))
for c, mk in zip(lag.columns[1:], 'osd^vx'): ax.plot(lag.index, lag[c], marker=mk, label=c.replace('r ', ''))
ax.set_xlabel('độ trễ k (ngày)'); ax.set_ylabel('tương quan với Q đến CĐ(t)'); ax.legend(fontsize=7, ncol=2); ax.set_xticks(range(11))
plt.tight_layout(); plt.savefig(OUT / 'fig04_do_tre_tuong_quan.png'); plt.show()

### 4.5 Quan hệ theo mùa và theo trạng thái dòng chảy (2019–2023)

Tương quan giữa X(t − k) và log Q CĐ(t), k = 0 … 3, tách theo mùa (lũ T7–T10 / kiệt) và theo trạng thái dòng chảy Cửa Đạt
(thấp < Q50, trung bình Q50–Q90, cao ≥ Q90 của 2019–2023). Bảng thứ hai: tương quan giữa **biến động** Δ log Q ra HN(t − k) và
Δ log Q CĐ(t) theo mùa. Đây là tương quan trên số liệu trung bình ngày — **không** xác định được chính xác thời gian truyền lũ
(nhỏ hơn 1 ngày không phân giải được); tương quan đồng thời (k = 0) có thể phản ánh cả mưa chung trên hai lưu vực.

In [ ]:
from src.data import hydro_analysis as ha
st_tab, (q50, q90) = ha.by_season_state(df, P1_END)
display(st_tab.round(3)); st_tab.to_csv(OUT / 'tuong_quan_theo_mua_trang_thai.csv', encoding='utf-8-sig')
dl = ha.delta_relation(df, P1_END); print('Δ log Q ra HN(t − k) với Δ log Q CĐ(t):'); display(dl.round(3)); dl.to_csv(OUT / 'bien_dong_q_ra_hn.csv', encoding='utf-8-sig')
fig, axs = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, v in zip(axs, ['Q đến HN', 'Q ra HN', 'Mưa V2']):
    t_ = st_tab.xs(v, level='biến')[['k = 0', 'k = 1', 'k = 2', 'k = 3']]
    for g, row in t_.iterrows(): ax.plot(range(4), row.values, marker='o', label=g)
    ax.set_title(v); ax.set_xlabel('độ trễ k (ngày)'); ax.set_xticks(range(4))
axs[0].set_ylabel('tương quan với log Q CĐ(t)'); axs[2].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / 'fig05_tuong_quan_mua_trang_thai.png'); plt.show()

### 4.6 Mưa và dòng chảy theo mùa lũ (tháng 7–10)

In [ ]:
fl = df[df.index.month.isin([7, 8, 9, 10])]; g = fl.groupby(fl.index.year)
yr = pd.DataFrame({'mưa V1 T7–T10 (mm)': g.mua_v1.sum(), 'mưa V1 ngày max': g.mua_v1.max(), 'mưa V2 T7–T10 (mm)': g.mua_v2.sum(),
                   'mưa V2 ngày max': g.mua_v2.max(), 'Q đến CĐ max': g.q_cd.max(), 'ngày thiếu Q CĐ': g.cd_missing.sum(), 'Q ra HN max': g.hn_qxa.max()}).round(0)
display(yr); yr.to_csv(OUT / 'mua_lu_theo_nam.csv', encoding='utf-8-sig')

### 4.7 Thống kê mô tả theo mùa (chỉ ngày có quan trắc)

Đặc điểm dòng chảy Cửa Đạt, dòng chảy đến và xả Hủa Na, mực nước Hủa Na và mưa vùng theo mùa lũ (T7–T10) / mùa kiệt (T11–T6).
Giai đoạn phân tích quan hệ 2019–2023 (như mục 4.1–4.5) và cả chuỗi 2019–08/2026 (chỉ để mô tả; không dùng cho bất kỳ lựa
chọn nào của mô hình). "Q ra − Q đến HN" > 0: hồ Hủa Na xả nhiều hơn lượng nước đến (tháo nước hồ); < 0: hồ tích nước.

In [ ]:
V = {'Q đến CĐ (m³/s)': ('q_cd', 'cd_missing'), 'Q đến HN (m³/s)': ('hn_qden', 'hn_missing'), 'Q ra HN (m³/s)': ('hn_qxa', 'hn_missing'),
     'Q ra − Q đến HN (m³/s)': ('hn_bal_raw', 'hn_missing'), 'Z HN (m)': ('hn_mn', 'hn_missing'), 'Mưa V1 (mm/ngày)': ('mua_v1', None), 'Mưa V2 (mm/ngày)': ('mua_v2', None)}
d_ = df.assign(hn_bal_raw=df.hn_qxa - df.hn_qden)
rows = []
for per, end in [('2019–2023', P1_END), ('2019–08/2026', df.index[-1])]:
    for mua, months in [('mùa lũ T7–T10', [7, 8, 9, 10]), ('mùa kiệt T11–T6', [11, 12, 1, 2, 3, 4, 5, 6])]:
        sub = d_.loc[:end]; sub = sub[sub.index.month.isin(months)]
        for nm, (c, miss) in V.items():
            x = sub[c][~sub[miss]] if miss else sub[c]
            x = x.dropna()
            rows.append({'giai đoạn': per, 'mùa': mua, 'biến': nm, 'n': len(x), 'TB': x.mean(), 'ĐLC': x.std(), 'min': x.min(),
                         'Q50': x.quantile(.5), 'Q95': x.quantile(.95), 'max': x.max()})
mt = pd.DataFrame(rows).set_index(['giai đoạn', 'mùa', 'biến'])
display(mt.round(2)); mt.to_csv(OUT / 'thong_ke_mo_ta_theo_mua.csv', encoding='utf-8-sig')

### 4.8 Kiểm tra cân bằng nước giữa hai hồ (chỉ mô tả)

Cửa Đạt nằm hạ lưu Hủa Na nên, về trung bình dài hạn, Q đến Cửa Đạt phải lớn hơn Q ra Hủa Na một lượng bằng dòng chảy gia nhập
khu giữa (V2). Bảng tính trung bình theo năm và mùa trên **cùng các ngày** có số liệu của cả hai hồ: "khu giữa" = Q đến CĐ − Q ra HN.
Giá trị âm kéo dài là dấu hiệu bất nhất giữa hai nguồn số liệu. Đơn vị cung cấp xác nhận cả hai chuỗi là số liệu đo (13/11/2026),
nên **nguyên nhân chưa giải thích được** (có thể Q ra HN thiếu thành phần xả khác, VX-04); dữ liệu và nhãn được giữ nguyên.

In [ ]:
c = df[~df.cd_missing & ~df.hn_missing].copy(); c['khu giữa'] = c.q_cd - c.hn_qxa
c['năm'] = c.index.year; c['mùa'] = np.where(c.index.month.isin([7, 8, 9, 10]), 'lũ T7–T10', 'kiệt T11–T6')
cb = c.groupby(['năm', 'mùa']).agg(**{'số ngày': ('q_cd', 'size'), 'Q đến CĐ TB': ('q_cd', 'mean'), 'Q ra HN TB': ('hn_qxa', 'mean'),
     'Q đến HN TB': ('hn_qden', 'mean'), 'khu giữa TB (m³/s)': ('khu giữa', 'mean'), '% ngày Q đến CĐ < Q ra HN': ('khu giữa', lambda x: 100 * (x < 0).mean()),
     'mưa V2 TB (mm/ngày)': ('mua_v2', 'mean')})
cb.loc[('cả chuỗi', ''), :] = [len(c), c.q_cd.mean(), c.hn_qxa.mean(), c.hn_qden.mean(), c['khu giữa'].mean(), 100 * (c['khu giữa'] < 0).mean(), c.mua_v2.mean()]
display(cb.round(1)); cb.to_csv(OUT / 'can_bang_nuoc_khu_giua.csv', encoding='utf-8-sig')